# Notebook for correlation and association plots
Needs to be run after `preprocessing.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path
from typing import cast

import pandas as pd

from utils.corr_assoc_utils import create_corr_assoc_matrix
from utils.misc_utils import turn_cats_to_continuous
from utils.plotting_utils import make_plot_corr_assoc_matrix
from utils.saving_utils import DatasetBundle, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Read in data
data_reduced = DatasetBundle.load(Path("data/preprocessed/data_discov_reduced"))
data_reduced.meta = cast(pd.DataFrame, data_reduced.meta)

data_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
data_full.meta = cast(pd.DataFrame, data_full.meta)

# Set outputs folder
output_folder = Path("outputs/corr_assoc_plots")
output_folder.mkdir(exist_ok=True)

In [ ]:
# Select variables from reduced data
groups = ["alcohol"]  # Change to one or more of: "alcohol", "nicotine", "drugs", "medication"

# Add vars from full data
full_data_groups = ["misc", "socioecon"]

vars_plot = [
    *data_reduced.meta.loc[lambda x: x["group"].isin(groups), "name"],
    *data_full.meta.loc[lambda x: x["group"].isin(full_data_groups), "name"],
]
vars_plot = [var for var in vars_plot if var not in ["fall", "zeit"]]  # Remove uninteresting variables

In [ ]:
# Treat binary and ordinal variables as if they were continuous (with equally spaced category coding)
treat_cat_as_continuous = data_full.meta.loc[lambda x: x["type"].isin(["binary", "ordinal"]), "name"]
data_plot = turn_cats_to_continuous(data_full.data[vars_plot], treat_cat_as_continuous)

In [ ]:
corr_assoc_matrix, p_value_matrix = create_corr_assoc_matrix(data_plot)
corr_assoc_matrix

In [ ]:
plot_corr_assoc_matrix = make_plot_corr_assoc_matrix(data_plot)
plot_corr_assoc_matrix.show()

save_plot(plot_corr_assoc_matrix, output_folder / ("plot_corr_assoc_matrix_" + "_".join(groups) + ".png"), dpi=150)